In [129]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
from sklearn.decomposition import PCA

In [28]:
!pip install "protobuf>=6.31.1"

Rúbrica: 

A continuación se muestra la rúbrica con la que se va a corregir el examen:


| Apartado/Criterio | Ponderación | 
| :-- | --- | 
| Ej. 1. Ha tratado de manera adecuada los datos de las columnas. | 0,5 | 
| Ej. 1. Ha usado como entrada las columnas adecuadas. | 0,5 |
| Ej. 1. Ha diseñado bien la red, el sistema de entrenamiento y ha comprobado si el resultado es bueno | 0,5 | 
| Ej. 1. Ha dimensionado bien la red neuronal. | 0,5 | 
| Ej. 1. Ha usado adecuadamente las técnicas de deep learning. | 2 | 
| Ej. 1. Ha tratado de realizar una predicción de forma adecuada. | 1,5 | 
| Ej. 2. Ha cargado los datos probando los dos métodos propuestos. | 1,5 | 
| Ej. 2. Ha diseñado bien la red convolucional y el sistema de entrenamiento | 1,5 | 
| Ej. 2. Ha dimensionado bien la red neuronal. | 0,5 | 
| Ej. 2. Ha usado las técnicas y el proceso visto en clase. | 0,5 | 
| Ej. 2. Ha usado su experiencia para valorar si el resultado es válido o no. | 0,5 | 



# EJERCICIO 1

In [29]:
empleo = pd.read_csv("destruccion_empleo.csv")

empleo.head()

,record_id,country,iso3_code,region,income_group,year,quarter,quarter_label,industry_sector,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced,data_source_notes
0,1,United States,USA,North America,High Income,2020,1,2020-Q1,Technology & Software,0.382,63514,0.0406,Research-calibrated synthetic data. Grounded i...
1,2,United States,USA,North America,High Income,2020,1,2020-Q1,Finance & Banking,0.608,63514,0.0517,Research-calibrated synthetic data. Grounded i...
2,3,United States,USA,North America,High Income,2020,1,2020-Q1,Healthcare & Life Sciences,0.198,63514,0.0176,Research-calibrated synthetic data. Grounded i...
3,4,United States,USA,North America,High Income,2020,1,2020-Q1,Manufacturing & Industry,0.720,63514,0.0924,Research-calibrated synthetic data. Grounded i...
4,5,United States,USA,North America,High Income,2020,1,2020-Q1,Retail & E-Commerce,0.676,63514,0.0667,Research-calibrated synthetic data. Grounded i...


In [30]:
empleo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   record_id                       20800 non-null  int64  
 1   country                         20800 non-null  object 
 2   iso3_code                       20800 non-null  object 
 3   region                          20800 non-null  object 
 4   income_group                    20800 non-null  object 
 5   year                            20800 non-null  int64  
 6   quarter                         20800 non-null  int64  
 7   quarter_label                   20800 non-null  object 
 8   industry_sector                 20800 non-null  object 
 9   sector_automation_risk_score    20800 non-null  float64
 10  gdp_per_capita_usd              20800 non-null  int64  
 11  pct_sector_workforce_displaced  20800 non-null  float64
 12  data_source_notes               

In [31]:
empleo.isnull().sum()

record_id                         0
country                           0
iso3_code                         0
region                            0
income_group                      0
year                              0
quarter                           0
quarter_label                     0
industry_sector                   0
sector_automation_risk_score      0
gdp_per_capita_usd                0
pct_sector_workforce_displaced    0
data_source_notes                 0
dtype: int64

In [32]:
columnas = empleo.select_dtypes(include=["object", "category"]).columns

for col in columnas:
    print(f"valores unicos en {col}:")
    print(empleo[col].unique())
    print("-" * 20)

valores unicos en country:
['United States' 'China' 'Germany' 'United Kingdom' 'Japan' 'France'
 'India' 'Canada' 'South Korea' 'Australia' 'Brazil' 'Italy' 'Spain'
 'Mexico' 'Indonesia' 'Netherlands' 'Saudi Arabia' 'Turkey' 'Switzerland'
 'Sweden' 'Norway' 'Denmark' 'Finland' 'Belgium' 'Austria' 'Poland'
 'Argentina' 'Colombia' 'Chile' 'Peru' 'South Africa' 'Nigeria' 'Egypt'
 'Kenya' 'Ethiopia' 'Ghana' 'Morocco' 'Tanzania' 'Pakistan' 'Bangladesh'
 'Vietnam' 'Thailand' 'Malaysia' 'Philippines' 'Singapore' 'New Zealand'
 'Portugal' 'Czech Republic' 'Romania' 'Hungary' 'Greece' 'Ukraine'
 'Israel' 'UAE' 'Qatar' 'Kuwait' 'Iran' 'Iraq' 'Kazakhstan' 'Uzbekistan'
 'Russia' 'Taiwan' 'Hong Kong' 'Sri Lanka' 'Nepal' 'Myanmar' 'Cambodia'
 'Ecuador' 'Bolivia' 'Paraguay' 'Uruguay' 'Costa Rica' 'Panama'
 'Guatemala' 'Algeria' 'Tunisia' 'Cameroon' 'Ivory Coast' 'Senegal'
 'Zambia']
--------------------
valores unicos en iso3_code:
['USA' 'CHN' 'DEU' 'GBR' 'JPN' 'FRA' 'IND' 'CAN' 'KOR' 'AUS' 'BRA' 'I

In [35]:
columnas_numericas = empleo.select_dtypes(include=["int64", "float64"]).columns

for col in columnas_numericas:
    print(f"valores unicos en {col}:")
    print(empleo[col].unique())
    print("-" * 20)

valores unicos en year:
[2020 2021 2022 2023 2024 2025 2026]
--------------------
valores unicos en quarter:
[1 2 3 4]
--------------------
valores unicos en sector_automation_risk_score:
[0.382 0.608 0.198 0.72  0.676 0.287 0.694 0.565 0.793 0.443 0.368 0.638
 0.205 0.756 0.75  0.278 0.715 0.557 0.797 0.405 0.415 0.605 0.226 0.753
 0.721 0.242 0.665 0.537 0.791 0.436 0.395 0.255 0.744 0.679 0.295 0.648
 0.566 0.779 0.472 0.367 0.639 0.265 0.712 0.692 0.687 0.577 0.792 0.466
 0.403 0.582 0.25  0.743 0.651 0.303 0.677 0.571 0.773 0.419 0.376 0.602
 0.233 0.723 0.672 0.243 0.661 0.55  0.763 0.465 0.383 0.61  0.203 0.728
 0.669 0.261 0.714 0.502 0.77  0.47  0.374 0.615 0.21  0.752 0.289 0.724
 0.568 0.42  0.359 0.594 0.201 0.78  0.685 0.253 0.674 0.807 0.477 0.394
 0.618 0.738 0.328 0.666 0.558 0.777 0.439 0.344 0.704 0.283 0.68  0.536
 0.81  0.449 0.36  0.599 0.228 0.739 0.696 0.279 0.684 0.564 0.463 0.629
 0.254 0.731 0.658 0.276 0.655 0.796 0.467 0.378 0.227 0.271 0.811 0.406
 0.38  0.

In [ ]:
empleo = empleo.drop(columns=["record_id", "iso3_code", "quarter_label", "data_source_notes"])


In [34]:
empleo.head()

,country,region,income_group,year,quarter,industry_sector,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced
0,United States,North America,High Income,2020,1,Technology & Software,0.382,63514,0.0406
1,United States,North America,High Income,2020,1,Finance & Banking,0.608,63514,0.0517
2,United States,North America,High Income,2020,1,Healthcare & Life Sciences,0.198,63514,0.0176
3,United States,North America,High Income,2020,1,Manufacturing & Industry,0.720,63514,0.0924
4,United States,North America,High Income,2020,1,Retail & E-Commerce,0.676,63514,0.0667


In [36]:
empleo = pd.get_dummies(empleo, dtype=int)

In [41]:
empleo.head()

,year,quarter,sector_automation_risk_score,gdp_per_capita_usd,pct_sector_workforce_displaced,country_Algeria,country_Argentina,country_Australia,country_Austria,country_Bangladesh,...,industry_sector_Administrative & Clerical,industry_sector_Education & Research,industry_sector_Energy & Utilities,industry_sector_Finance & Banking,industry_sector_Healthcare & Life Sciences,industry_sector_Manufacturing & Industry,industry_sector_Media & Communications,industry_sector_Retail & E-Commerce,industry_sector_Technology & Software,industry_sector_Transportation & Logistics
0,2020,1,0.382,63514,0.0406,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,2020,1,0.608,63514,0.0517,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,2020,1,0.198,63514,0.0176,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,2020,1,0.720,63514,0.0924,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,2020,1,0.676,63514,0.0667,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [37]:
empleo.corr()["pct_sector_workforce_displaced"].abs().sort_values(ascending=False)[1:]

gdp_per_capita_usd                  0.641091
income_group_High Income            0.572740
sector_automation_risk_score        0.477408
year                                0.450259
income_group_Lower Middle Income    0.374951
                                      ...   
country_Hungary                     0.008761
quarter                             0.008421
country_Poland                      0.007610
country_China                       0.004474
country_Greece                      0.003983
Name: pct_sector_workforce_displaced, Length: 110, dtype: float64

In [44]:
X = empleo.drop(["pct_sector_workforce_displaced"], axis=1)
y = empleo["pct_sector_workforce_displaced"]

In [46]:
escalador = StandardScaler()
X = escalador.fit_transform(X)
y = escalador.fit_transform(y.to_frame())

In [47]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)

In [48]:
X_train.shape[1:]

(110,)

Random Forest para poder encaminarnos mejor en las redes neuronales.

In [49]:
rnd_reg = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
rnd_reg.fit(X_train, y_train)

/home/ciabd01/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [50]:
y_pred = rnd_reg.predict(X_test)

In [51]:
r2_score(y_test, y_pred)

0.9577280408138305

In [ ]:
# La red que da mejores resultados
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(60, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(20, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(1))

W0000 00:00:1778516008.404956   19328 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [64]:
# Primera prueba con Nadam

model.compile(loss='mean_squared_error', optimizer= keras.optimizers.Nadam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['mae'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

In [65]:
history = model.fit(X_train, y_train, epochs=150, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=32)

Epoch 1/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0775 - mae: 0.2078 - val_loss: 0.0536 - val_mae: 0.1742
Epoch 2/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0758 - mae: 0.2023 - val_loss: 0.0455 - val_mae: 0.1449
Epoch 3/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0739 - mae: 0.2014 - val_loss: 0.0480 - val_mae: 0.1530
Epoch 4/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0727 - mae: 0.2010 - val_loss: 0.0432 - val_mae: 0.1433
Epoch 5/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0717 - mae: 0.1978 - val_loss: 0.0508 - val_mae: 0.1505
Epoch 6/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0746 - mae: 0.2037 - val_loss: 0.0438 - val_mae: 0.1434
Epoch 7/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0718 - mae: 0.1988 - val_loss: 0.0455 - val_mae: 0.1506
Epoch 8/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0721 - mae: 0.2005 - val_loss: 0.0483 - val_mae: 0.1519
Epoch 9/150
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/

In [66]:
model.evaluate(X_test, y_test)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step - loss: 0.0452 - mae: 0.1434


[0.045169949531555176, 0.14341211318969727]

In [ ]:
# Segunda prueba con Adam y con más paciencia porque se "corta" muy pronto, ademas de añadir mas learning rate.
# Da mejores resultados con learning rate mas bajo, más paciencia y con el optimizar Adam y no Nadam.

model.compile(loss='mean_squared_error', optimizer= keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['mae'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)

In [61]:
history = model.fit(X_train, y_train, epochs=300, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=32)

Epoch 1/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0661 - mae: 0.1935 - val_loss: 0.0434 - val_mae: 0.1433
Epoch 2/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 984us/step - loss: 0.0667 - mae: 0.1935 - val_loss: 0.0402 - val_mae: 0.1367
Epoch 3/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0608 - mae: 0.1830 - val_loss: 0.0406 - val_mae: 0.1377
Epoch 4/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step - loss: 0.0631 - mae: 0.1882 - val_loss: 0.0407 - val_mae: 0.1399
Epoch 5/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 986us/step - loss: 0.0667 - mae: 0.1942 - val_loss: 0.0401 - val_mae: 0.1362
Epoch 6/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 993us/step - loss: 0.0590 - mae: 0.1823 - val_loss: 0.0406 - val_mae: 0.1371
Epoch 7/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0669 - mae: 0.1927 - val_loss: 0.0415 - val_mae: 0.1412
Epoch 8/300
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0637 - mae: 0.1868 - val_loss: 0.0409 - val_mae: 0.1380
Epoch 9/300
422/422 ━━━━━━━━━━━━━━━━━━━━

In [62]:
model.evaluate(X_test, y_test)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 733us/step - loss: 0.0426 - mae: 0.1365


[0.04256248474121094, 0.13652504980564117]

In [ ]:
# Cambio el numero de parametros a más:

model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(70, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(30, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(1))

In [103]:

model.compile(loss='mean_squared_error', optimizer= keras.optimizers.Nadam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['mae'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

In [108]:
# Compruebo que haciendo ese ajuste de la red, el loss se pasa.
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=32)

Epoch 1/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 2.8620e-04 - mae: 0.0033 - val_loss: 4.2598e-05 - val_mae: 0.0042
Epoch 2/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 9.9144e-07 - mae: 5.4362e-04 - val_loss: 1.9201e-05 - val_mae: 0.0032
Epoch 3/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 9.2524e-06 - mae: 0.0017 - val_loss: 9.2125e-06 - val_mae: 0.0018
Epoch 4/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.9467e-05 - mae: 0.0017 - val_loss: 3.7485e-05 - val_mae: 0.0039
Epoch 5/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.2683e-05 - mae: 0.0026 - val_loss: 3.6335e-04 - val_mae: 0.0143
Epoch 6/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.4161e-05 - mae: 0.0022 - val_loss: 1.0338e-05 - val_mae: 0.0030
Epoch 7/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.9171e-05 - mae: 0.0028 - val_loss: 2.4463e-05 - val_mae: 0.0032
Epoch 8/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 3.4913e-05 - mae: 0.0034 - val_loss: 1.2235e-0

In [109]:
model.evaluate(X_test, y_test)

29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0366e-07 - mae: 2.9538e-04 


[3.036620910279453e-07, 0.00029538123635575175]

In [105]:
# Cambio el numero de parametros a menos

model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(15, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(1))

In [110]:

model.compile(loss='mean_squared_error', optimizer= keras.optimizers.Nadam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['mae'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

In [111]:
# El loss se sigue pasando si disminuyo la red, la mejor es la 1º red, con 60,20
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=32)

Epoch 1/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 4.8999e-05 - mae: 0.0014 - val_loss: 2.3871e-04 - val_mae: 0.0102
Epoch 2/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.9266e-05 - mae: 0.0028 - val_loss: 1.7840e-04 - val_mae: 0.0074
Epoch 3/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.0695e-05 - mae: 0.0026 - val_loss: 8.5140e-05 - val_mae: 0.0054
Epoch 4/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3.2798e-05 - mae: 0.0036 - val_loss: 2.7258e-04 - val_mae: 0.0108
Epoch 5/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 2.4669e-05 - mae: 0.0027 - val_loss: 7.0964e-06 - val_mae: 0.0016
Epoch 6/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3.0786e-05 - mae: 0.0029 - val_loss: 5.6090e-05 - val_mae: 0.0047
Epoch 7/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.5389e-05 - mae: 0.0034 - val_loss: 4.3177e-06 - val_mae: 0.0020
Epoch 8/500
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.6692e-05 - mae: 0.0032 - val_loss: 1.1276e-05 - 

In [112]:
model.evaluate(X_test, y_test)

29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.7718e-07 - mae: 5.3357e-04 


[5.771802875642607e-07, 0.0005335687892511487]

In [ ]:
# Prediccion



# EJERCICIO 2

In [180]:
# No recuerdo como tratar los datos sin que esten separados por carpetas, las creo y separo.

folders = listdir('./DeFungi')

photos = []
labels = []

for idx, folder in enumerate(folders):
    for file in listdir('./DeFungi/'+ folder):
        photo = load_img('./DeFungi/'+folder+'/'+file, color_mode='grayscale', target_size=(117,117))
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0
1
2
3
4


In [197]:
photos_array = asarray(photos)
labels_array = asarray(labels)


In [198]:
X = np.asarray(photos)
y = np.asarray(labels)

In [199]:
X.shape

(9114, 117, 117, 1)

In [196]:
#X = X.reshape(9114, -1)

In [200]:
X = X/255.0

In [201]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)

In [ ]:
# Se prueba un random forest para encaminarnos, nos da un 0.54, con una red neuronal convolucional deberiamos de sacar mas.

rnd_clas = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rnd_clas.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [ ]:
y_pred = rnd_clas.predict(X_test)

In [189]:
accuracy_score(y_test, y_pred)

0.5427631578947368

In [203]:
x = photos_array/255.0

In [204]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, random_state=42)


In [205]:
X_train.shape[1:]

(117, 117, 1)

In [ ]:
# Primera red neuronal - primera prueba

model = keras.models.Sequential()

model.add(Conv2D(16,(3,3), activation='relu', input_shape=(117,117, 1)))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Flatten())

model.add(keras.layers.Dense(150, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='softmax', kernel_initializer='glorot_normal'))

In [213]:
model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [214]:
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=256)

Epoch 1/500
26/26 ━━━━━━━━━━━━━━━━━━━━ 17s 606ms/step - accuracy: 0.5208 - loss: 1.2765 - val_accuracy: 0.4881 - val_loss: 1.3161
Epoch 2/500
26/26 ━━━━━━━━━━━━━━━━━━━━ 17s 654ms/step - accuracy: 0.5792 - loss: 1.0339 - val_accuracy: 0.4881 - val_loss: 1.5165
Epoch 3/500
26/26 ━━━━━━━━━━━━━━━━━━━━ 17s 659ms/step - accuracy: 0.5993 - loss: 0.9630 - val_accuracy: 0.4881 - val_loss: 2.3118
Epoch 4/500
26/26 ━━━━━━━━━━━━━━━━━━━━ 18s 681ms/step - accuracy: 0.6255 - loss: 0.8973 - val_accuracy: 0.4881 - val_loss: 2.1598
Epoch 5/500
26/26 ━━━━━━━━━━━━━━━━━━━━ 17s 668ms/step - accuracy: 0.6533 - loss: 0.8399 - val_accuracy: 0.4881 - val_loss: 2.4910
Epoch 6/500
26/26 ━━━━━━━━━━━━━━━━━━━━ 18s 679ms/step - accuracy: 0.6883 - loss: 0.7721 - val_accuracy: 0.5009 - val_loss: 2.0888


In [ ]:
# Claramente gana esta red, con el optimizar Adam, cambiando solamente vas abajo el optimizar a Nadam, da un accuracy mucho menor.
model.evaluate(X_test, y_test)

29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4803 - loss: 1.3043


[1.304260492324829, 0.4802631437778473]

In [259]:
# Segunda red - segunda prueba
model = keras.models.Sequential()

model.add(Conv2D(16,(3,3), activation='relu', input_shape=(117,117, 1)))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Flatten())

model.add(keras.layers.Dense(150, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='softmax', kernel_initializer='glorot_normal'))

In [260]:
model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Nadam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [261]:
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=256)

Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 638ms/step - accuracy: 0.3663 - loss: 1.5878 - val_accuracy: 0.2143 - val_loss: 1.6507
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 634ms/step - accuracy: 0.4964 - loss: 1.2010 - val_accuracy: 0.2063 - val_loss: 1.7504
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 670ms/step - accuracy: 0.5548 - loss: 1.0683 - val_accuracy: 0.2286 - val_loss: 3.7462
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 692ms/step - accuracy: 0.6468 - loss: 0.8783 - val_accuracy: 0.2254 - val_loss: 7.2450
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 742ms/step - accuracy: 0.7341 - loss: 0.7199 - val_accuracy: 0.2254 - val_loss: 5.6513
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 716ms/step - accuracy: 0.8690 - loss: 0.4236 - val_accuracy: 0.2254 - val_loss: 11.5833


In [262]:
model.evaluate(X_test, y_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1657 - loss: 1.7012


[1.7011603116989136, 0.16571427881717682]

In [272]:
# Tercera red - con mas neuronas en cada.
model = keras.models.Sequential()

model.add(Conv2D(16,(3,3), activation='relu', input_shape=(117,117, 1)))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Flatten())

model.add(keras.layers.Dense(300, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='softmax', kernel_initializer='glorot_normal'))

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [273]:
model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [274]:
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=256)

Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 673ms/step - accuracy: 0.3567 - loss: 1.6515 - val_accuracy: 0.1857 - val_loss: 4.6526
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 723ms/step - accuracy: 0.4754 - loss: 1.2495 - val_accuracy: 0.2587 - val_loss: 3.5795
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 719ms/step - accuracy: 0.5147 - loss: 1.1666 - val_accuracy: 0.2254 - val_loss: 1.8157
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 647ms/step - accuracy: 0.5683 - loss: 1.0745 - val_accuracy: 0.2143 - val_loss: 1.7865
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 644ms/step - accuracy: 0.6175 - loss: 0.9596 - val_accuracy: 0.2048 - val_loss: 5.8959
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 636ms/step - accuracy: 0.6817 - loss: 0.8256 - val_accuracy: 0.2032 - val_loss: 9.2756
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 646ms/step - accuracy: 0.7429 - loss: 0.6906 - val_accuracy: 0.2032 - val_loss: 8.0164
Epoch 8/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 634ms/step - accuracy: 0.7913 - loss: 0.5520 - val_accu

In [ ]:
# Sigue sin mejorar si se añade mas neuronas.
model.evaluate(X_test, y_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1886 - loss: 1.7144


[1.7143830060958862, 0.18857142329216003]

In [263]:
# Bajando el learning rate con la misma red
model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Nadam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [264]:
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=256)

Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 715ms/step - accuracy: 0.4063 - loss: 1.4440 - val_accuracy: 0.1984 - val_loss: 1.7761
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 661ms/step - accuracy: 0.5246 - loss: 1.1393 - val_accuracy: 0.2063 - val_loss: 1.7157
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 640ms/step - accuracy: 0.6099 - loss: 0.9658 - val_accuracy: 0.1698 - val_loss: 1.8622
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 632ms/step - accuracy: 0.7036 - loss: 0.7631 - val_accuracy: 0.1825 - val_loss: 2.0600
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 615ms/step - accuracy: 0.8437 - loss: 0.4646 - val_accuracy: 0.1794 - val_loss: 2.1726
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 635ms/step - accuracy: 0.9294 - loss: 0.2476 - val_accuracy: 0.2429 - val_loss: 2.4653
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 641ms/step - accuracy: 0.9679 - loss: 0.1253 - val_accuracy: 0.2413 - val_loss: 3.2843


In [265]:
model.evaluate(X_test, y_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2200 - loss: 1.6892


[1.6892496347427368, 0.2199999988079071]

In [ ]:
# Cuarta red - segunda prueba - añadiendo una capa más y cambiando el optimizar a Adam. Además al añadir una capa más, se añaden mas neuronas, 300, 150, 50
model = keras.models.Sequential()

model.add(Conv2D(16,(3,3), activation='relu', input_shape=(117,117, 1)))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Flatten())

model.add(keras.layers.Dense(300, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(150, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='softmax', kernel_initializer='glorot_normal'))

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [267]:
model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [268]:
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=256)

Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 621ms/step - accuracy: 0.3488 - loss: 1.6360 - val_accuracy: 0.1857 - val_loss: 10.1726
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 656ms/step - accuracy: 0.4746 - loss: 1.2400 - val_accuracy: 0.2016 - val_loss: 4.8679
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 704ms/step - accuracy: 0.5111 - loss: 1.1598 - val_accuracy: 0.1810 - val_loss: 5.3075
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 717ms/step - accuracy: 0.5484 - loss: 1.0604 - val_accuracy: 0.2873 - val_loss: 1.6993
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 735ms/step - accuracy: 0.6091 - loss: 0.9509 - val_accuracy: 0.2444 - val_loss: 2.5460
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 677ms/step - accuracy: 0.6603 - loss: 0.8223 - val_accuracy: 0.2302 - val_loss: 2.9870
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 670ms/step - accuracy: 0.7067 - loss: 0.7458 - val_accuracy: 0.2381 - val_loss: 2.9474
Epoch 8/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 658ms/step - accuracy: 0.7552 - loss: 0.6106 - val_a

In [ ]:
# Se comprueba que añadiendo una capa más sube un poco el accuracy pero en ningun momento llega al punto de la primera red con una capa menos.
model.evaluate(X_test, y_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.2457 - loss: 1.7373


[1.7372772693634033, 0.24571429193019867]

In [226]:
# Recogiendo menos datos y haciendo las imagenes mas pequeñas
# Como lo he separado en carpetas, me he fijado y minimo en cada una hay 700, entonces intento recoger 700 fotos de cada subcarpeta.

folders = listdir('./DeFungi')

photos = []
labels = []

for idx, folder in enumerate(folders):
    for file in listdir('./DeFungi/'+ folder)[:700]:
        photo = load_img('./DeFungi/'+folder+'/'+file, color_mode='grayscale', target_size=(64,64))
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0
1
2
3
4


In [227]:
photos_array = asarray(photos)
labels_array = asarray(labels)

In [228]:
X = np.asarray(photos)
y = np.asarray(labels)

In [229]:
X.shape

(3500, 64, 64, 1)

In [230]:
X = X/255.0

In [231]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)

In [232]:
x = photos_array/255.0

In [233]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, random_state=42)

In [234]:
X_train.shape[1:]

(64, 64, 1)

In [235]:
model = keras.models.Sequential()

model.add(Conv2D(16,(3,3), activation='relu', input_shape=(64,64, 1)))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Flatten())

model.add(keras.layers.Dense(150, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='softmax', kernel_initializer='glorot_normal'))

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [236]:
model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Nadam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [237]:
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=128)

Epoch 1/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.4044 - loss: 1.4436 - val_accuracy: 0.2016 - val_loss: 1.8002
Epoch 2/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.5000 - loss: 1.1683 - val_accuracy: 0.1873 - val_loss: 2.0198
Epoch 3/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.5571 - loss: 1.0467 - val_accuracy: 0.1857 - val_loss: 3.9269
Epoch 4/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.6437 - loss: 0.8885 - val_accuracy: 0.1857 - val_loss: 4.7401
Epoch 5/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step - accuracy: 0.7218 - loss: 0.7253 - val_accuracy: 0.1857 - val_loss: 7.0905
Epoch 6/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 0.8155 - loss: 0.5072 - val_accuracy: 0.1857 - val_loss: 5.6694


In [ ]:
# La conslusion reducciendo la imagen de tamaño y el numero de fotos es que da un accuracy muy bajo comparado con la primera, en la que se recoge todo el numero de fotos que hay.
model.evaluate(X_test, y_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2029 - loss: 1.7205 


[1.7204543352127075, 0.2028571367263794]

In [239]:
# Recogiendo menos datos pero con el mismo tamaño que recogi la primera vez 117,117
# Como lo he separado en carpetas, me he fijado y minimo en cada una hay 700, entonces intento recoger 700 fotos de cada subcarpeta.

folders = listdir('./DeFungi')

photos = []
labels = []

for idx, folder in enumerate(folders):
    for file in listdir('./DeFungi/'+ folder)[:700]:
        photo = load_img('./DeFungi/'+folder+'/'+file, color_mode='grayscale', target_size=(117,117))
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0
1
2
3
4


In [243]:
photos_array = asarray(photos)
labels_array = asarray(labels)

In [244]:
X = np.asarray(photos)
y = np.asarray(labels)

In [245]:
X.shape

(3500, 117, 117, 1)

In [246]:
X = X/255.0

In [247]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)

In [248]:
x = photos_array/255.0

In [249]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, random_state=42)

In [250]:
X_train.shape[1:]

(117, 117, 1)

In [251]:
model = keras.models.Sequential()

model.add(Conv2D(16,(3,3), activation='relu', input_shape=(117,117, 1)))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(32,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(2,2))

model.add(Flatten())

model.add(keras.layers.Dense(150, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(50, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='softmax', kernel_initializer='glorot_normal'))

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [252]:
# Cambio a Adam y subo el learning rate ya que son menos imagenes.

model.compile(loss='sparse_categorical_crossentropy', optimizer= keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

In [253]:
# Tambien reduzco el batch size al ser menos imagenes.
history = model.fit(X_train, y_train, epochs=500, validation_split=0.2, callbacks=[early_stopping_cb], batch_size=128)

Epoch 1/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 329ms/step - accuracy: 0.3925 - loss: 1.5220 - val_accuracy: 0.2032 - val_loss: 1.9674
Epoch 2/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 363ms/step - accuracy: 0.5667 - loss: 1.0980 - val_accuracy: 0.2254 - val_loss: 2.5152
Epoch 3/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 358ms/step - accuracy: 0.6877 - loss: 0.8662 - val_accuracy: 0.2254 - val_loss: 4.0890
Epoch 4/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 345ms/step - accuracy: 0.7988 - loss: 0.6095 - val_accuracy: 0.2254 - val_loss: 4.6479
Epoch 5/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 346ms/step - accuracy: 0.9056 - loss: 0.3966 - val_accuracy: 0.2254 - val_loss: 5.2538
Epoch 6/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 353ms/step - accuracy: 0.9532 - loss: 0.2421 - val_accuracy: 0.2254 - val_loss: 5.9926


In [ ]:
# La conslusion que se saca es que reduciendo el numero de imagenes a 3500, da un accuracy muy bajo, comparado sin la reducción que supera el 0.40
model.evaluate(X_test, y_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1971 - loss: 2.0606


[2.06064772605896, 0.1971428543329239]